<a href="https://colab.research.google.com/github/donleaveher/plasticity-placement/blob/agent%2Fadd-lora-evaluation/notebooks/p0c_confirmatory_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P0-C Confirmatory: 24 Lessons × 3 Seeds

This notebook is the standalone Tier 2 entry point. It requires a completed calibration report with a non-null `selected_config`, resolves the exact calibration code revision, and writes every confirmatory attempt to its own Google Drive directory.

When a run has an immutable failed unit, diagnose it and change `CONFIRMATORY_RUN` to a new directory name. Do not delete or overwrite the failed run.

## 1. Mount Drive and choose run directories

Edit only `CALIBRATION_RUN` and `CONFIRMATORY_RUN`. The calibration directory must contain the successful report used for the frozen experiment. Use a fresh confirmatory directory name for each new attempt.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import subprocess
from collections import Counter
from hashlib import sha256
from pathlib import Path

REPO_URL = 'https://github.com/donleaveher/plasticity-placement.git'
BRANCH = 'agent/add-lora-evaluation'
REPO_DIR = Path('/content/plasticity-placement')
DRIVE_ROOT = Path('/content/drive/MyDrive/plasticity-p0c')

# Change these two names for your Drive layout.
CALIBRATION_RUN = 'calibration-v3'
CONFIRMATORY_RUN = 'confirmatory-v3'

CALIBRATION_DIR = DRIVE_ROOT / CALIBRATION_RUN
CALIBRATION_REPORT = CALIBRATION_DIR / 'calibration_report.json'
CONFIRMATORY_DIR = DRIVE_ROOT / CONFIRMATORY_RUN

if CALIBRATION_DIR == CONFIRMATORY_DIR:
    raise ValueError('Calibration and confirmatory directories must be different')

print('Calibration report:', CALIBRATION_REPORT)
print('Confirmatory output:', CONFIRMATORY_DIR)

## 2. Validate calibration and resolve its code revision

This cell fails before creating a confirmatory run when the calibration report is missing, has `selected_config: null`, or lacks the environment snapshot needed to recover its exact Git commit.

In [ ]:
if not CALIBRATION_REPORT.exists():
    raise FileNotFoundError(f'Missing calibration report: {CALIBRATION_REPORT}')

calibration_report = json.loads(CALIBRATION_REPORT.read_text())
selected_config = calibration_report.get('selected_config')
if not isinstance(selected_config, dict) or not selected_config:
    raise RuntimeError(
        f'{CALIBRATION_REPORT} has no qualified selected_config; '
        'choose a successful calibration run'
    )

calibration_config = calibration_report.get('calibration_config')
provenance = calibration_report.get('provenance')
if not isinstance(calibration_config, dict) or not isinstance(provenance, dict):
    raise RuntimeError('Calibration report is missing config or provenance')

candidate_id = selected_config.get('candidate_id')
environment_candidates = [
    CALIBRATION_DIR / 'candidates' / str(candidate_id) / stage / 'environment.json'
    for stage in ('stage1', 'stage2')
]
environment_path = next((path for path in environment_candidates if path.exists()), None)
if environment_path is None:
    raise FileNotFoundError(
        'Cannot locate calibration environment.json for selected candidate '
        f'{candidate_id}'
    )

calibration_environment = json.loads(environment_path.read_text())
CALIBRATION_REVISION = calibration_environment.get('git_commit')
EXPECTED_CODE_HASH = provenance.get('code_sha256')
if not CALIBRATION_REVISION or not EXPECTED_CODE_HASH:
    raise RuntimeError('Calibration provenance lacks git commit or code hash')

print('Selected config:', selected_config)
print('Calibration revision:', CALIBRATION_REVISION)
print('Expected code hash:', EXPECTED_CODE_HASH)
print('Model revision:', provenance.get('resolved_model_revision'))

## 3. Checkout the frozen code and install dependencies

The repository is checked out in detached mode at the exact commit recorded by the successful calibration run. The code hash must match before any confirmatory artifacts are created.

In [ ]:
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['pip', 'install', '-q', 'uv'], check=True)

if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
subprocess.run(
    ['git', '-C', str(REPO_DIR), 'checkout', '--detach', str(CALIBRATION_REVISION)],
    check=True,
)
subprocess.run(
    ['uv', 'sync', '--extra', 'train', '--extra', 'colab'],
    cwd=REPO_DIR,
    check=True,
)

current_code_hash = subprocess.run(
    [
        'uv', 'run', 'python', '-c',
        'from plasticity_placement.p0c.runtime import current_code_hash; '
        'print(current_code_hash())',
    ],
    cwd=REPO_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

if current_code_hash != EXPECTED_CODE_HASH:
    raise RuntimeError(
        'Calibration/code hash mismatch: '
        f'expected={EXPECTED_CODE_HASH}, current={current_code_hash}'
    )

print('Pinned revision:', CALIBRATION_REVISION)
print('Verified code hash:', current_code_hash)

## 4. Freeze this confirmatory attempt

The notebook writes a small configuration snapshot before the run. Reusing the same directory with different calibration or code settings is rejected. A disconnected but otherwise healthy run may resume from the same directory.

In [ ]:
CONFIRMATORY_DIR.mkdir(parents=True, exist_ok=True)
notebook_config_path = CONFIRMATORY_DIR / 'notebook_config.json'
notebook_config = {
    'tier': 'confirmatory',
    'training_seeds': [41, 42, 43],
    'calibration_report': str(CALIBRATION_REPORT),
    'calibration_report_sha256': sha256(CALIBRATION_REPORT.read_bytes()).hexdigest(),
    'calibration_revision': str(CALIBRATION_REVISION),
    'code_sha256': current_code_hash,
}

if notebook_config_path.exists():
    existing_config = json.loads(notebook_config_path.read_text())
    if existing_config != notebook_config:
        raise RuntimeError(
            f'Existing confirmatory directory has different settings: '
            f'{notebook_config_path}'
        )
else:
    config_tmp = notebook_config_path.with_suffix('.json.tmp')
    config_tmp.write_text(json.dumps(notebook_config, indent=2, sort_keys=True) + '\n')
    config_tmp.replace(notebook_config_path)

print(json.dumps(notebook_config, indent=2))

## 5. Run Tier 2

Set `RUN_CONFIRMATORY = True` only after all previous checks pass. The run contains 24 lessons and three adapter seeds. Verified units resume after a disconnect; failed units remain immutable.

In [ ]:
run_command = [
    'uv', 'run', 'plasticity-p0c', 'run',
    '--output', str(CONFIRMATORY_DIR),
    '--tier', 'confirmatory',
    '--model', str(calibration_config['model_name']),
    '--max-length', str(calibration_config['max_length']),
    '--max-new-tokens', str(calibration_config['max_new_tokens']),
    '--seeds', '41', '42', '43',
    '--calibration-config', str(CALIBRATION_REPORT),
]
if calibration_config.get('use_4bit'):
    run_command.append('--use-4bit')
if calibration_config.get('model_revision') is not None:
    run_command.extend(['--model-revision', str(calibration_config['model_revision'])])

print('Command:', ' '.join(run_command))

RUN_CONFIRMATORY = False
if RUN_CONFIRMATORY:
    subprocess.run(run_command, cwd=REPO_DIR, check=True)

## 6. Progress and failure inspection

This cell is read-only. Run it after a disconnect or failure before deciding whether to resume the directory or start a new attempt.

In [ ]:
manifest_path = CONFIRMATORY_DIR / 'manifest.json'
screening_path = CONFIRMATORY_DIR / 'results' / 'screening.jsonl'

if not manifest_path.exists():
    print('No manifest yet; failure occurred during preflight or provenance validation.')
else:
    manifest = json.loads(manifest_path.read_text())
    print('Run ID:', manifest.get('run_id'))
    print('Selected lessons:', len(manifest.get('selected_lessons', [])))
    print('Base states:', Counter(manifest.get('base_arms', {}).values()))
    print('Unit states:', Counter(
        unit.get('state') for unit in manifest.get('units', {}).values()
    ))
    print('Recent errors:')
    for error in manifest.get('errors', [])[-10:]:
        print(error)

print('Screening artifact exists:', screening_path.exists())

## 7. Aggregate the complete run

Aggregation refuses incomplete seed/probe panels, non-verified units, mixed precision, provenance mismatches, or rollback below exact match 1.0.

In [ ]:
RUN_AGGREGATE = False
if RUN_AGGREGATE:
    subprocess.run([
        'uv', 'run', 'plasticity-p0c', 'aggregate',
        '--output', str(CONFIRMATORY_DIR),
        '--bootstrap-samples', '10000',
    ], cwd=REPO_DIR, check=True)

    summary_path = CONFIRMATORY_DIR / 'results' / 'aggregate' / 'summary.json'
    summary = json.loads(summary_path.read_text())
    print('Selected lessons:', summary['selected_lesson_count'])
    print('Probe rows:', summary['probe_row_count'])
    if summary['selected_lesson_count'] != 24 or summary['probe_row_count'] != 6864:
        raise RuntimeError('Unexpected confirmatory result dimensions')
    display(summary['arm_summary'])
    display(summary['contrasts'])

## Directory policy

- Resume the same `CONFIRMATORY_RUN` only after a normal disconnect and only when no unit is `failed`.
- If a unit is `failed`, preserve that directory, diagnose the recorded error, and increment `CONFIRMATORY_RUN` for the corrected attempt.
- If model, precision, calibration, compiler, or code changes, use a new calibration run and a new confirmatory run.
- Never edit `calibration_report.json`, manually replace `selected_config`, or use `--overwrite-compiled` for a formal run.